In [13]:
import numpy as np
from qutip import Qobj, tensor, basis
from qutip.qip.operations import rotation, cnot, cz

###############################################################################
#                              Gate Definitions                               #
###############################################################################

def hadamard_single_qubit():
    """
    Construct a single-qubit Hadamard gate using QuTiP's rotation() gates.
    Up to a global phase, H = Rz(pi/2) * Ry(pi/2).
    """
    # Rotation around Z by pi/2
    rz_pi_2 = rotation('Z', np.pi/2)
    # Rotation around Y by pi/2
    ry_pi_2 = rotation('Y', np.pi/2)
    # Combined
    return rz_pi_2 @ ry_pi_2

def hadamard(N, target_qubit):
    """
    Create the multi-qubit Hadamard on `target_qubit` for an N-qubit system.
    """
    # Single-qubit H
    h_single = hadamard_single_qubit()
    
    # We "tensor" identity on all other qubits
    ops = []
    for qubit_idx in range(N):
        if qubit_idx == target_qubit:
            ops.append(h_single)
        else:
            # Identity for a single qubit
            ops.append(Qobj(np.eye(2), dims=[[2],[2]]))
    return tensor(*ops)

def cnot_gate(N, control, target):
    """
    Returns the CNOT gate for an N-qubit system using QuTiP’s built-in cnot().
    """
    return cnot(N, control, target)

def cz_gate(N, control, target):
    """
    Returns the CZ (controlled-Z) gate for an N-qubit system using QuTiP’s built-in cz().
    """
    return cz(N, control, target)

###############################################################################
#                        Circuit Simulation Framework                         #
###############################################################################

def apply_circuit(gate_sequence, N=2, initial_state=None, return_state_vector=True):
    """
    Simulates a circuit of gates on an N-qubit system.
    
    Parameters
    ----------
    gate_sequence : list of tuples
        Each element describes a gate and the qubits it acts on.
        Example formats:
          - ("H", target)
          - ("CNOT", control, target)
          - ("CZ", control, target)
        where target, control are integer qubit indices.
    
    N : int
        Number of qubits in the system. Default 2.
    
    initial_state : qutip.Qobj or None
        If None, the system is initialized to |0...0>.
        If provided, should be a ket (state vector) or a density matrix.
        Must have the correct dimension (2^N).
    
    return_state_vector : bool
        If True and the final state is pure, return a NumPy 1D vector of size 2^N.
        Otherwise, return a QuTiP `Qobj`.
    
    Returns
    -------
    final_state : np.ndarray or qutip.Qobj
        The final state after applying the gates.
    """
    dim = 2**N
    
    # -------------------------------------------------------------------------
    # 1. Set up the initial state
    # -------------------------------------------------------------------------
    if initial_state is None:
        # Start in |0...0>
        ket0 = basis(2, 0)
        init_state_qobj = tensor(*[ket0 for _ in range(N)])  # |0>^{\otimes N}
    else:
        # Ensure this is a Qobj with the right dimension
        if not isinstance(initial_state, Qobj):
            raise TypeError("initial_state must be a QuTiP Qobj or None.")
        if initial_state.shape != (dim, dim) and initial_state.shape != (dim, 1):
            raise ValueError(f"initial_state dimension must be {dim}x1 or {dim}x{dim}.")
        init_state_qobj = initial_state
    
    # -------------------------------------------------------------------------
    # 2. Build (or apply) each gate in sequence
    # -------------------------------------------------------------------------
    current_state = init_state_qobj
    for gate_info in gate_sequence:
        name = gate_info[0].upper()
        
        if name == "H":
            # gate_info = ("H", target_qubit)
            _, target = gate_info
            gate_op = hadamard(N, target)
        
        elif name == "CNOT":
            # gate_info = ("CNOT", control, target)
            _, ctrl, targ = gate_info
            gate_op = cnot_gate(N, ctrl, targ)
        
        elif name == "CZ":
            # gate_info = ("CZ", control, target)
            _, ctrl, targ = gate_info
            gate_op = cz_gate(N, ctrl, targ)
        
        else:
            raise ValueError(f"Unknown gate name: {name}")
        
        # Apply this gate to the current state
        # If we have a density matrix (isoper), use U * rho * U^\dagger
        # If we have a ket, use U * ket
        if current_state.isoper:
            current_state = gate_op @ current_state @ gate_op.dag()
        else:
            current_state = gate_op @ current_state
    
    # -------------------------------------------------------------------------
    # 3. Return final state in requested format
    # -------------------------------------------------------------------------
    if return_state_vector and (not current_state.isoper):
        # Return as a 1D NumPy array
        return current_state.full().ravel()
    else:
        # Return as a QuTiP Qobj
        return current_state

###############################################################################
#                           Example Usage Demo                                 #
###############################################################################

if __name__ == "__main__":
    # Example gate sequence on a 2-qubit system:
    # 1. Hadamard on qubit 0
    # 2. CNOT(control=0, target=1)
    gate_seq = [
        ("H", 0),
        ("CNOT", 0, 1)
    ]
    
    final = apply_circuit(gate_seq, N=2, initial_state=None, return_state_vector=True)
    print("Final state as vector:", final)
    
    # Another example: Apply CZ after the same operations
    gate_seq_cz = [
        ("H", 0),
        ("CNOT", 0, 1),
        ("CZ", 0, 1)
    ]
    final_cz = apply_circuit(gate_seq_cz, N=2, return_state_vector=True)
    print("Final state (with CZ) as vector:", final_cz)


ImportError: cannot import name 'cz' from 'qutip.qip.operations' (/Users/runzhaoguo/miniconda3/envs/qc/lib/python3.10/site-packages/qutip_qip/operations/__init__.py)